# Compositional (indirect) noise and the compact choked nozzle

A composition inhomogeneity — a pocket of gas with a different mixture, hence a different $\gamma$ and $R$ — radiates sound when it is accelerated through a nozzle, exactly as an entropy (temperature) spot does.
This is **compositional**, or **indirect**, combustion noise (Magri, *JFM* 2016).

A choked nozzle can be closed three ways in FNS, and they are *not* equivalent for this mechanism:

1. **The hand-written compact closure** `PerturbationBC.choked_nozzle()` — the textbook Marble–Candel formula $g = R\,f + R_s\,h$, a 3-wave $(f,g,h)$ relation.
2. **The inherited compact element** `choked_nozzle_outlet(A*)` — the critical-mass-flux jump, linearized **by complex step** through its *full* state dependence.
3. **A resolved nozzle** — area-change elements and ducts, with the near-throat remainder lumped.

For the **acoustic** reflection $R$ the three agree in the compact limit.
But the hand-written closure carries only $(f,g,h)$ — it has **no composition column** $R_\xi$ — so it silently drops compositional noise.
Routes 2 and 3 linearize the *actual* jump, so the same complex step that gives $R$ and the entropy coupling $R_s$ also gives $R_\xi$ **for free**.

This notebook validates the acoustic limit, demonstrates the compositional noise the inherited element captures, and shows the limitation of the hand-written closure.
A closing note places it against the $M=1$ subsonic scope.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from fns.elements import catalog as cat
from fns.perturbation.operator.boundary_bc import PerturbationBC
from fns.perturbation import forced_response, CompositionalNoiseWarning
from fns.perturbation.operator.operator import build_acoustic_blocks, assemble_acoustic
from fns.perturbation.operator.characteristics import char_to_dx, edge_caloric
from fns.solver import solve
from fns.solver.control import states_table
from fns.thermo.configure import perfect_gas, equilibrium
from fns.thermo.api import EQ_FROZEN
from fns.assembly.derive import ES_RHO, ES_C, ES_U, ES_P, ES_AREA, ES_M
from fns.plotting import use_fns_theme, COLORWAY

use_fns_theme()
R_AIR, GAMMA = 287.0, 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)
GM1 = GAMMA - 1.0

## 1. Inert acoustic limit — the three terminations agree

A high-pressure reservoir discharges through a feed duct to a choked nozzle (perfect-gas air).
We drive a unit acoustic wave at the inlet and read the reflection $g/f$ at the **application plane** (the duct head, just upstream of the nozzle).

* the **compact** termination is the inherited `choked_nozzle_outlet(A*)`;
* the **resolved** termination resolves part of the contraction (an `isentropic_area_change` and a throat duct) and lumps only the near-throat remainder.

Both stay strictly subsonic on the application plane — the sonic point lives *inside* the lumped throat.

In [ ]:
PT, TT, A_IN, A_STAR = 1.6e5, 300.0, 0.05, 0.03
FREQ = np.linspace(1.0, 1500.0, 300)  # Hz

DRIVE_ACOUSTIC = PerturbationBC.anechoic(driven=("acoustic",))  # unit acoustic wave in, absorb returning sound
DRIVE_ENTROPY = PerturbationBC.anechoic(driven=("entropy",))    # unit entropy wave in, absorb returning sound


def build_compact(inlet, analytic=False):
    """Feed duct -> compact choked nozzle (inherited, or the hand-written analytic closure)."""
    nb = PerturbationBC.choked_nozzle() if analytic else None
    els = [cat.total_pressure_inlet(PT, TT, perturbation_bc=inlet),
           cat.duct(0.6),
           cat.choked_nozzle_outlet(A_STAR, perturbation_bc=nb)]
    prob = cat.build_problem(perfect_gas(R_AIR, GAMMA), els, [(0, 1, A_IN), (1, 2, A_IN)], 10.0, 1e5, CP * TT)
    return prob, solve(prob).x


def build_resolved(inlet, A_mid=0.04):
    """Feed duct -> isentropic contraction -> throat duct -> lumped choked throat."""
    els = [cat.total_pressure_inlet(PT, TT, perturbation_bc=inlet),
           cat.duct(0.6),
           cat.isentropic_area_change(),
           cat.duct(0.25),
           cat.choked_nozzle_outlet(A_STAR)]
    edges = [(0, 1, A_IN), (1, 2, A_IN), (2, 3, A_mid), (3, 4, A_mid)]
    prob = cat.build_problem(perfect_gas(R_AIR, GAMMA), els, edges, 10.0, 1e5, CP * TT)
    return prob, solve(prob).x


def reflection(prob, x, plane=1):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return forced_response(prob, x, FREQ).reflection_at(plane)


prob_c, x_c = build_compact(DRIVE_ACOUSTIC)
prob_a, x_a = build_compact(DRIVE_ACOUSTIC, analytic=True)
prob_r, x_r = build_resolved(DRIVE_ACOUSTIC)
R_compact = reflection(prob_c, x_c)
R_analytic = reflection(prob_a, x_a)
R_resolved = reflection(prob_r, x_r)

M_plane = float(states_table(prob_c, x_c)[ES_M, 1])
R_mc = (2.0 - GM1 * M_plane) / (2.0 + GM1 * M_plane)
print(f"application-plane Mach M = {M_plane:.3f}")
print(f"Marble-Candel R(M)       = {R_mc:.4f}")
print(f"inherited vs analytic closure  max|dR| = {np.max(np.abs(R_compact - R_analytic)):.2e}  (identical to round-off)")

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(r"$|R(f)|$", r"$\angle R(f)\;[\mathrm{deg}]$"),
                    horizontal_spacing=0.11)
for y, name, col in [(np.abs(R_compact), "compact (inherited)", COLORWAY[0]),
                     (np.abs(R_resolved), "resolved", COLORWAY[2])]:
    fig.add_trace(go.Scatter(x=FREQ, y=y, name=name, line=dict(color=col)), row=1, col=1)
fig.add_hline(y=R_mc, line=dict(dash="dash", color="#52606d"),
              annotation_text="Marble-Candel R(M)", annotation_position="bottom right", row=1, col=1)
for y, name, col in [(np.degrees(np.angle(R_compact)), "compact (inherited)", COLORWAY[0]),
                     (np.degrees(np.angle(R_resolved)), "resolved", COLORWAY[2])]:
    fig.add_trace(go.Scatter(x=FREQ, y=y, name=name, line=dict(color=col), showlegend=False), row=1, col=2)
fig.update_xaxes(title_text=r"$f\;[\mathrm{Hz}]$", row=1, col=1)
fig.update_xaxes(title_text=r"$f\;[\mathrm{Hz}]$", row=1, col=2)
fig.update_layout(height=420, width=960, title_text="Compact vs resolved choked nozzle (inert)")
fig.show()

The inherited compact element and the hand-written analytic closure give the **same** $R$ to round-off — the complex-step linearization of the critical-mass-flux jump *is* the Marble–Candel formula, no hand-coded coefficient needed.

The compact $|R|$ is **flat** at $R(M)$: a compact nozzle's reflection depends only on the application-plane Mach.
The resolved nozzle coincides with it as $f\to 0$ (the section is acoustically compact there) and departs at higher frequency, where the finite length of the resolved contraction adds its own propagation phase.
Compactness is precisely the **low-frequency** limit of the resolved nozzle.

### Entropy noise — the same comparison

The reflection $R$ is only one entry of the nozzle's scattering.
The off-diagonal $R_s$ — an arriving **entropy** (temperature) spot radiating sound as it accelerates — is *indirect (entropy) noise*, and it compares compact-vs-resolved in exactly the same spirit.
We drive a unit **entropy** wave at the (anechoic) inlet and read the sound it radiates back upstream, $g$, at the application plane.

(The same comparison against the De Domenico *et al.* (2019) compact model — acoustic **and** entropic transfer functions of a lossy nozzle — is the validation in `entropy_generator.ipynb` and `tests/test_perturbation_dedomenico.py`.)

In [ ]:
def upstream_sound(prob, x, plane=1):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return forced_response(prob, x, FREQ).waves(plane)[:, 1]  # g: the upstream-radiated wave


prob_ce, x_ce = build_compact(DRIVE_ENTROPY)
prob_re, x_re = build_resolved(DRIVE_ENTROPY)
g_compact = upstream_sound(prob_ce, x_ce)
g_resolved = upstream_sound(prob_re, x_re)

fig = make_subplots(rows=1, cols=2, subplot_titles=(r"$|g(f)|$  (entropy noise)", r"$\angle g(f)\;[\mathrm{deg}]$"),
                    horizontal_spacing=0.11)
for y, name, col in [(np.abs(g_compact), "compact (inherited)", COLORWAY[0]),
                     (np.abs(g_resolved), "resolved", COLORWAY[2])]:
    fig.add_trace(go.Scatter(x=FREQ, y=y, name=name, line=dict(color=col)), row=1, col=1)
for y, col in [(np.degrees(np.angle(g_compact)), COLORWAY[0]), (np.degrees(np.angle(g_resolved)), COLORWAY[2])]:
    fig.add_trace(go.Scatter(x=FREQ, y=y, line=dict(color=col), showlegend=False), row=1, col=2)
fig.update_xaxes(title_text=r"$f\;[\mathrm{Hz}]$", row=1, col=1)
fig.update_xaxes(title_text=r"$f\;[\mathrm{Hz}]$", row=1, col=2)
fig.update_layout(height=420, width=960, title_text="Entropy noise: compact vs resolved choked nozzle (inert)")
fig.show()
print(f"compact  |g|: {np.abs(g_compact).min():.2f}..{np.abs(g_compact).max():.2f}  (flat -> constant R_s)")
print(f"resolved |g|: {np.abs(g_resolved).min():.2f}..{np.abs(g_resolved).max():.2f}")

Same verdict, with one extra twist worth noting.
The **magnitude** story is identical to the reflection: compact $|g|$ is **flat** (a single $R_s$), and the resolved nozzle meets it at $f\to 0$ then departs.
But the **phase** winds far faster than the reflection's, because the entropy spot rides the slow **convective** speed $u$ across the nozzle (a delay $L/u$), not the acoustic $L/(c\pm u)$ — so the resolved entropy noise becomes non-compact at a *lower* frequency than the reflection does.
That convective phase is the heart of the non-compact entropy-noise corrections (De Domenico; Duran–Moreau).

## 2. Compositional noise — what the inherited linearization captures

Now make the flow **reacting and multi-stream**: air enters, hydrogen is injected, and the mixture (frozen here, so it is a pure *composition* spot, not a flame) convects to the choked nozzle.
Each feed stream is a transported mixture fraction $\xi$; a fluctuation $\xi'$ changes the local $\gamma,R$ and therefore the critical mass flux.

We read the nozzle's outlet row in characteristic coordinates,

$$ g = R\,f + R_s\,h + R_\xi\cdot\boldsymbol{\xi}, $$

extracting $R$ (acoustic reflection), $R_s$ (entropy noise) and $R_\xi$ (compositional noise) directly from the linearized row.

In [ ]:
from thermolib import SpeciesLibrary, Thermo

MECH = os.path.join("..", "thermolib", "data", "h2o2.yaml")
AIR = {"O2": 0.21, "N2": 0.79}
lib = SpeciesLibrary.from_native(MECH)
gas = Thermo(lib)
_idx = lib.species_index
_Y = np.zeros(lib.n_species); _Y[_idx["O2"]], _Y[_idx["N2"]] = 0.21, 0.79; _Y /= _Y.sum()
H_AIR = gas.enthalpy_mass(_Y, 300.0)


def reacting_nozzle(nozzle_bc, inlet_bc=None, mdot_h2=0.02):
    """Air inlet -> duct -> H2 source -> duct -> compact choked nozzle (frozen composition spot)."""
    els = [cat.mass_flow_inlet(0.4, 300.0, composition=AIR, basis="mole", perturbation_bc=inlet_bc),
           cat.duct(0.4),
           cat.mass_source(mdot_h2, 300.0, composition={"H2": 1.0}, basis="mole"),
           cat.duct(0.5),
           cat.choked_nozzle_outlet(0.012, perturbation_bc=nozzle_bc)]
    edges = [(0, 1, 0.02), (1, 2, 0.02), (2, 3, 0.02), (3, 4, 0.02)]
    return cat.build_problem(equilibrium(gas.mech), els, edges, 0.4, 1e5, H_AIR, edge_models=[EQ_FROZEN] * 4)


def nozzle_row(prob, x, stamped):
    """(M, R, R_s, R_xi) of the choked-nozzle outlet row in (f, g, h, xi) coordinates."""
    est = states_table(prob, x)
    K = float(prob.tf[0]) / float(prob.tf[1])
    cal = edge_caloric(prob, x)[3]
    ns, e, r0 = int(prob.n_solve), 3, int(prob.node_row_ptr[4])
    if stamped:  # an explicit closure overwrites the row
        A = assemble_acoustic(0.0, build_acoustic_blocks(prob, x), with_boundaries=True).tocsr()
        row = np.asarray(A[r0, ns * e: ns * e + ns].todense()).ravel()
    else:        # the inherited J_alg row
        row = np.asarray(build_acoustic_blocks(prob, x).J_alg[r0, ns * e: ns * e + ns].todense()).ravel()
    rho, c, u, p, area, M = (float(est[k, e]) for k in (ES_RHO, ES_C, ES_U, ES_P, ES_AREA, ES_M))
    cf, cg, ch = row[:3] @ char_to_dx(rho, c, u, p, area, K, cal)
    return M, -cf / cg, -ch / cg, -row[3:] / cg


prob_in = reacting_nozzle(None)
x_in = solve(prob_in).x
streams = list(prob_in.scalar_names)
M, R, R_s, R_xi = nozzle_row(prob_in, x_in, stamped=False)
print("feed streams (mixture fractions):", streams)
print(f"nozzle approach Mach M = {M:.3f}")
print(f"R   (acoustic)      = {R.real:+.4f}    [Marble-Candel {(2 - GM1 * M)/(2 + GM1 * M):.4f}]")
print(f"R_s (entropy noise) = {R_s.real:+.4g}")
for nm, v in zip(streams, R_xi):
    print(f"R_xi[{nm!r}] (compositional noise) = {v.real:+.2f}")

The inherited row reproduces the Marble–Candel **acoustic** reflection $R$, and at the same time carries a **non-zero** composition column $R_\xi$ — large for the hydrogen `source` stream, whose mixture fraction strongly moves $\gamma$ and $R$.
That column *is* the compositional noise: the same complex step that produced $R$ and $R_s$ produced it.

To see it radiate, drive a pure composition wave at the inflow (no acoustic, no entropy forcing) and read the reflected wave $g$ at the nozzle.

In [ ]:
fuel = streams[-1]  # the injected hydrogen-stream mixture fraction
prob_d = reacting_nozzle(None, inlet_bc=PerturbationBC.anechoic(driven=(fuel,)))
x_d = solve(prob_d).x
with warnings.catch_warnings():
    warnings.simplefilter("error", CompositionalNoiseWarning)  # inherited element -> no gap, no warning
    fr = forced_response(prob_d, x_d, FREQ)
xi_in = fr.waves(0)[:, fr.wave_labels.index(fuel)]   # the composition wave at the inflow
g_noz = fr.waves(3)[:, 1]                              # reflected acoustic wave at the nozzle

fig = go.Figure()
fig.add_trace(go.Scatter(x=FREQ, y=np.abs(xi_in), name=r"$|\xi'|$ in (drive)", line=dict(color=COLORWAY[3])))
fig.add_trace(go.Scatter(x=FREQ, y=np.abs(g_noz), name=r"$|g|$ radiated at nozzle", line=dict(color=COLORWAY[0])))
fig.update_layout(height=400, width=820, xaxis_title=r"$f\;[\mathrm{Hz}]$",
                  title="A pure composition wave radiating sound at the choked nozzle")
fig.show()
print(f"unit composition wave in -> |g| out = {np.abs(g_noz).min():.1f} .. {np.abs(g_noz).max():.1f}")

A unit composition wave alone produces a substantial reflected acoustic wave — generated entirely by the inherited linearization, with no hand-written scalar closure.
This is the indirect-noise analogue of the entropy-noise chain, driven by composition instead of temperature.

## 3. The limitation — the hand-written compact closure drops it

Swap the inherited element for the hand-written `PerturbationBC.choked_nozzle()` closure and read the same row.
The closure now takes the **backend-correct** effective $\gamma = \rho c^2/p$ from the state, so its acoustic reflection $R$ matches the inherited value.
But it overwrites the row with a 3-wave $(f,g,h)$ relation: its composition column is **identically zero**.
The structural gap is the missing $R_\xi$ — there is simply no slot for a composition fluctuation in the closed form.

In [ ]:
prob_an = reacting_nozzle(PerturbationBC.choked_nozzle())
x_an = solve(prob_an).x
Ma, Ra, R_sa, R_xia = nozzle_row(prob_an, x_an, stamped=True)
print(f"analytic closure:  R = {Ra.real:+.4f}   (inherited R = {R.real:+.4f} -- matches: correct gamma)")
print(f"                   R_s = {R_sa.real:+.4g}")
for nm, v in zip(streams, R_xia):
    print(f"  R_xi[{nm!r}] = {v.real:+.2e}   (dropped)")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    forced_response(prob_an, x_an, np.array([200.0]))
print("\nCompositionalNoiseWarning fired:",
      any(issubclass(w.category, CompositionalNoiseWarning) for w in caught))

In [ ]:
labels = [f"R_xi[{s}]" for s in streams]
fig = go.Figure()
fig.add_trace(go.Bar(x=labels, y=np.abs(R_xi), name="inherited element", marker_color=COLORWAY[0]))
fig.add_trace(go.Bar(x=labels, y=np.abs(R_xia), name="analytic closure", marker_color=COLORWAY[4]))
fig.update_layout(height=400, width=720, barmode="group",
                  yaxis_title=r"$|R_\xi|$  (compositional-noise coupling)",
                  title="Compositional noise: captured (inherited) vs dropped (analytic closure)")
fig.show()

The hand-written closure carries the **entropy** off-diagonal $R_s$ (entropy noise) but has no composition column at all.
That is exactly the case the `CompositionalNoiseWarning` flags: a compact analytic nozzle closure on a reacting flow.

**The takeaway:** for compositional noise at a compact nozzle, use the **inherited** `choked_nozzle_outlet` element, or **resolve** the nozzle.
Both linearize the real jump and get $R_\xi$ from the same complex step that gives $R$ and $R_s$ — no new formula, no validation case of its own.

## 4. The $M = 1$ boundary

Everything here stays strictly **subsonic** on the application plane — FNS v1 scope.
At $M=1$ the upstream acoustic characteristic has zero speed: its duct transit time $\tau_- = L/(c-u)\to\infty$, the boundary-value problem changes type, and the isentropic area–Mach relation turns over.
This is a genuine singularity of the sonic point, not a coding choice.

The **compact** closure is precisely the way around it at a boundary: it imposes the sonic condition *algebraically*, lumping the sonic point **outside** the domain so the application plane stays subsonic.
The **resolved** route works arbitrarily close to choke ($M\to 1^-$) but not at the point itself.
Propagating the acoustic field *through* a sonic throat needs a different solver — integrating the linearized-Euler equations through the $M(x)$ profile with the sonic-regularity (L'Hôpital) condition (Stow–Dowling / Duran–Moreau).
That distributed, through-throat nozzle response — the proper home for non-compact compositional noise — is deferred with the supersonic scope.